In [ ]:
### GET PATHS samples 

In [1]:
# -----######-----###### STRICT 2-LEVEL FILE PATH EXTRACTOR -----######-----###### #
import os
import pandas as pd
from tqdm import tqdm

def _paths_1707_strict2_GET_df(folder_root):
    """
    Collect file paths exactly two folder levels deep inside folder_root.
    Example: folder_root/level1/level2/*.*
    """
    all_paths = []

    level1 = [os.path.join(folder_root, d1) for d1 in os.listdir(folder_root)
              if os.path.isdir(os.path.join(folder_root, d1)) and not d1.startswith(".")]

    for path1 in tqdm(level1, desc="📂 Level 1"):
        level2 = [os.path.join(path1, d2) for d2 in os.listdir(path1)
                  if os.path.isdir(os.path.join(path1, d2)) and not d2.startswith(".")]

        for path2 in level2:
            for file in os.listdir(path2):
                if not file.startswith(".") and not file.startswith("._"):
                    full_path = os.path.join(path2, file)
                    if os.path.isfile(full_path):
                        all_paths.append(full_path)

    return pd.DataFrame({'Path': all_paths})


In [7]:
folder_root = "/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S/_25_07_VOCAL"
df = _paths_1707_strict2_GET_df(folder_root)
len(df)


📂 Level 1: 100%|███████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 50.38it/s]


13514

In [12]:
# -----######-----###### FINAL SMART PARSER (lu[A-Z] generalized) -----######-----###### #

import os
import re
import pandas as pd
from tqdm import tqdm

def _stem_1710_nodict_GET_df_keys(path_list):
    rows = []

    for path in tqdm(path_list, desc="🎯 Parsing Stem Filenames"):
        base = os.path.basename(path)
        name = os.path.splitext(base)[0]

        # Get BPM from end
        bpm_match = re.search(r'-(\d{2,3})\.mp3$', base)
        bpm_sample = bpm_match.group(1) if bpm_match else None

        # Get id_stem_sample like s13_v3-7110z
        id_match = re.search(r'_s\d+_(v\d+-[0-9a-zA-Z]+)', name)
        id_stem_sample = id_match.group(0)[1:] if id_match else None
        id_stem = id_match.group(1).split("-")[1] if id_match else None

        # Extract key string using generic 'luX' marker
        key_string = ""
        try:
            # Remove extension and bpm
            clean_name = name.replace(f"-{bpm_sample}", "") if bpm_sample else name

            # Match the last 'luX-' or 'luX_' style
            lu_match = re.search(r'(lu[A-Z])[-_]([\w\-#min#maj_]+)$', clean_name)
            if lu_match:
                key_string = lu_match.group(2)
        except:
            pass

        # Break into tokens
        tokens = re.split(r'[-_]', key_string)

        # Extract DJ–Music key pairs
        pairs = []
        i = 0
        while i < len(tokens) - 1:
            dj = tokens[i]
            mus = tokens[i+1]
            if re.match(r'^\d{1,2}[AB]$', dj):
                pairs.append((dj, mus))
                i += 2
            else:
                i += 1

        # Unpack pairs
        key_dj_1, key_1 = pairs[0] if len(pairs) > 0 else (None, None)
        key_dj_2, key_2 = pairs[1] if len(pairs) > 1 else (None, None)
        key_dj_3, key_3 = pairs[2] if len(pairs) > 2 else (None, None)

        rows.append({
            "Path": path,
            "base_name": name,
            "id_stem": id_stem,
            "id_stem_sample": id_stem_sample,
            "bpm_sample": bpm_sample,
            "key_string": key_string,
            "key_1": key_1,
            "key_dj_1": key_dj_1,
            "key_2": key_2,
            "key_dj_2": key_dj_2,
            "key_3": key_3,
            "key_dj_3": key_dj_3
        })

    return pd.DataFrame(rows)


In [13]:
df_keys = _stem_1710_nodict_GET_df_keys(df['Path'])
df_keys

🎯 Parsing Stem Filenames: 100%|████████████████████████████████████| 13514/13514 [00:00<00:00, 116646.79it/s]


,Path,base_name,id_stem,id_stem_sample,bpm_sample,key_string,key_1,key_dj_1,key_2,key_dj_2,key_3,key_dj_3
0,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s13_v4-7130b-3BC#maj-luO-10B-Dmaj-3B-C#maj_12...,7130b,s13_v4-7130b,123,10B-Dmaj-3B-C#maj_12A-C#min,Dmaj,10B,C#maj,3B,C#min,12A
1,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s21_v4-7130b-3BC#maj-luO-3B-C#maj-11A-F#min_1...,7130b,s21_v4-7130b,123,3B-C#maj-11A-F#min_10B-Dmaj,C#maj,3B,F#min,11A,Dmaj,10B
2,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_sa2_v4-7130b-3BC#maj-luO-4A-Fmin-3B-C#maj_12A...,None,None,123,4A-Fmin-3B-C#maj_12A-C#min,Fmin,4A,C#maj,3B,C#min,12A
3,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_sa1_v4-7130b-3BC#maj-luO-12A-C#min-11A-F#min_...,None,None,123,12A-C#min-11A-F#min_3B-C#maj,C#min,12A,F#min,11A,C#maj,3B
4,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s42_v4-7130b-3BC#maj-luO-8B-Cmaj-4A-Fmin_7B-F...,7130b,s42_v4-7130b,123,8B-Cmaj-4A-Fmin_7B-Fmaj,Cmaj,8B,Fmin,4A,Fmaj,7B
...,...,...,...,...,...,...,...,...,...,...,...,...
13509,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s44_v3-7110z-3AA#min-luM-1B-Bmaj-3A-A#min_2B-...,7110z,s44_v3-7110z,123,1B-Bmaj-3A-A#min_2B-F#maj,Bmaj,1B,A#min,3A,F#maj,2B
13510,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s13_v3-7110z-3AA#min-luM-10A-Bmin-3A-A#min_2B...,7110z,s13_v3-7110z,123,10A-Bmin-3A-A#min_2B-F#maj,Bmin,10A,A#min,3A,F#maj,2B
13511,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s42_v3-7110z-3AA#min-luM-6B-A#maj-10B-Dmaj_7A...,7110z,s42_v3-7110z,123,6B-A#maj-10B-Dmaj_7A-Dmin,A#maj,6B,Dmaj,10B,Dmin,7A
13512,/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATER...,_s14_v3-7110z-3AA#min-luM-7B-Fmaj-4A-Fmin_11A-...,7110z,s14_v3-7110z,123,7B-Fmaj-4A-Fmin_11A-F#min,Fmaj,7B,Fmin,4A,F#min,11A


In [18]:
print(pd.concat([df_keys[c] for c in ['base_name','id_stem','id_stem_sample','bpm_sample','key_1','key_dj_1','key_2','key_dj_2','key_3','key_dj_3']], axis=1).to_string())


                                                                                          base_name id_stem id_stem_sample bpm_sample  key_1 key_dj_1  key_2 key_dj_2  key_3 key_dj_3
0                                         _s13_v4-7130b-3BC#maj-luO-10B-Dmaj-3B-C#maj_12A-C#min-123   7130b   s13_v4-7130b        123   Dmaj      10B  C#maj       3B  C#min      12A
1                                         _s21_v4-7130b-3BC#maj-luO-3B-C#maj-11A-F#min_10B-Dmaj-123   7130b   s21_v4-7130b        123  C#maj       3B  F#min      11A   Dmaj      10B
2                                          _sa2_v4-7130b-3BC#maj-luO-4A-Fmin-3B-C#maj_12A-C#min-123    None           None        123   Fmin       4A  C#maj       3B  C#min      12A
3                                        _sa1_v4-7130b-3BC#maj-luO-12A-C#min-11A-F#min_3B-C#maj-123    None           None        123  C#min      12A  F#min      11A  C#maj       3B
4                                             _s42_v4-7130b-3BC#maj-luO-8B-Cmaj-4A-Fmin_7B